# Gratimos · codegen

**Regenerate a module and your hand edits are still there.** A genuine conflict raises instead of silently overwriting somebody's work — which is precisely why architecture-as-code needs a merge and not a template.

> **Every cell in this notebook runs.** They are generated from
> [`tools/notebooks/spec.py`](../tools/notebooks/spec.py) and executed by CI, so a
> cell that cannot run does not reach a commit. Change a cell, re-run it, and the
> page is yours — that is what it is for.


Template-based generators have one failure mode and everybody has met it: you
edit the generated file, somebody regenerates, and your edit is gone. The usual
workaround is a `DO NOT EDIT` banner, which relocates the problem to "then where
*do* I put the custom logic".

This merges at the **AST** level, three ways: the previous generation, the
current file on disk, and the new generation. Reformatting is invisible to it —
it compares trees, not text.

In [ ]:
# --- setup: works locally, on Binder, and on Colab -------------------------
import subprocess, sys, pathlib

def _ensure_installed():
    """Make the package importable, preferring the checkout this notebook is in.

    The checkout comes first deliberately. Trusting whichever `slpie` happens to
    be installed means a notebook opened inside one clone can silently exercise
    a different one — which is exactly what happened while writing this.
    """
    here = pathlib.Path.cwd()
    root = next(
        (p for p in [here, *here.parents] if (p / "pyproject.toml").exists()), None,
    )
    if root is None:                     # Colab: no checkout, so fetch one
        root = pathlib.Path("/content/Macropol-s")
        if not root.exists():
            subprocess.run(
                ["git", "clone", "--depth", "1",
                 "https://github.com/Reimain/Macropol-s.git", str(root)],
                check=True,
            )
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))
    try:
        import slpie, gratimos          # noqa: F401
    except ModuleNotFoundError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(root)],
                       check=True)
    return root

ROOT = _ensure_installed()
print("package root:", ROOT)

import slpie
print("slpie", slpie.__version__)

## Generate a module from a shape

In [ ]:
import pathlib, tempfile
from gratimos.codegen import ModuleRegistry
from gratimos.meta.infer import infer_shape

WORK = pathlib.Path(tempfile.mkdtemp(prefix="gratimos-nb-"))

shape = infer_shape(
    [{"id": 1, "customer": "Ada", "total": 99.5, "priority": True}],
    name="Orders",
)

registry_of_modules = ModuleRegistry(WORK)
generation = registry_of_modules.generate(shape)

module_path = WORK / "orders.py"
print(module_path.read_text())

## Now edit it by hand, the way a person would

In [ ]:
source = module_path.read_text()
source += (
    '\n\n'
    'def total_with_tax(order, rate: float = 0.2) -> float:\n'
    '    """Hand-written. Nobody generated this."""\n'
    '    return round(order.total * (1 + rate), 2)\n'
)
module_path.write_text(source)
print(module_path.read_text()[-240:])

## Regenerate, with the shape changed

In [ ]:
wider = infer_shape(
    [{"id": 1, "customer": "Ada", "total": 99.5, "priority": True,
      "discount": 0.1, "channel": "web"}],
    name="Orders",
)

registry_of_modules.generate(wider)
after = module_path.read_text()

print("the new fields arrived:")
print("  discount:", "discount" in after)
print("  channel: ", "channel" in after)
print()
print("and the hand-written function survived:")
print("  total_with_tax:", "total_with_tax" in after)

In [ ]:
print(after[-420:])

## Reformatting is invisible to the merge

It compares ASTs. Re-indent the whole file, change quote style, reflow an argument list — none of that is a change.

In [ ]:
from gratimos.codegen.astmerge import merge_sources

base   = "def f(a, b):\n    return a + b\n"
theirs = "def f(a, b):\n    return a+b     # reformatted only\n"
ours   = "def f(a, b, c=0):\n    return a + b + c\n"

result = merge_sources(ours, theirs, base)
print("conflicts:", len(result.conflicts))
print("decisions:", [d.name if hasattr(d, 'name') else str(d) for d in result.decisions][:4])
print()
print(result.source)

## A genuine conflict raises rather than guessing

In [ ]:
from gratimos.codegen.astmerge import ConflictPolicy
from gratimos.errors import MergeConflict

base   = "LIMIT = 100\n"
theirs = "LIMIT = 500\n"        # the generator wants 500
ours   = "LIMIT = 250\n"        # a human chose 250

try:
    merge_sources(ours, theirs, base, policy=ConflictPolicy.RAISE)
    print("(no conflict was detected for this pair)")
except MergeConflict as error:
    print("refused, and named what disagrees:")
    print(" ", error)

Silently taking either side would be wrong. Taking the generator's loses a deliberate human decision; taking the human's loses a real schema change. Raising is the only answer that does not discard information.

## `# gratimos:keep` makes it unconditional

In [ ]:
kept = module_path.read_text() + '\n\nDEBUG = True  # gratimos:keep\n'
module_path.write_text(kept)

registry_of_modules.generate(shape)     # regenerate with the *narrower* shape
final = module_path.read_text()

print("marked line survived a regeneration that did not know about it:",
      "DEBUG = True" in final)

## Where this is used in anger

This is the single Gratimos import SLPIE is allowed (`slpie/artifacts/codegen.py`, invariant 8, asserted by test). TOGAF views generate into `architecture/*.py` through exactly this path — so an architect's annotation survives the next scan.

In [ ]:
# Scratch cell — generate from your own shape and edit the result.
mine = infer_shape([{"sku": "A-1", "qty": 3}], name="Line")
registry_of_modules.generate(mine)
print((WORK / "line.py").read_text()[:400])